# 03 — Paired YOLO26n Training

Fine-tune the pretrained neck/head on matched Raw and Gamma-or-Skip S10 inputs using the same initialization, budget and Raw-development selection protocol. Native training owns optimization, checkpointing and per-epoch validation; no upstream preparation or separate baseline evaluation is invoked.

- **Inputs:** Prepared six-class protocol/subsets/policy/decision, paired S10 exports, Raw development export and pinned local `yolo26n.pt`.
- **Outputs:** `runs/training-s10-v1/`: named native runs, best/last checkpoints, histories, native results.csv and best.pt/last.pt; minimal paired dataset manifest.
- **Prerequisites:** Study environment, compatible CUDA hardware for recorded settings, prepared artifacts and a new native training root or explicit native resume.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Pretrained Initialization and Paired Inputs](#pretrained-paired-inputs)
3. [Raw Training](#raw-training)
4. [Adaptive Training](#adaptive-training)
5. [Convergence and Resource Inspection](#convergence-resources)
6. [Checkpoint and Integrity Summary](#checkpoint-integrity)

Set paths in section 1, then **Restart & Run All**. Only this phase executes.
Missing or mismatched required artifacts fail; upstream phases are never launched.
The historical six-class taxonomy, including railway `train`, stays unchanged.

<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

Select paths and explicit native per-branch resume choices. Missing inputs fail rather than invoking data/preprocessing notebooks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import read_json, require_paths
from vehicle_detection.finetuning import TRAIN_SETTINGS, prepare_pair, train_branch
from vehicle_detection.protocol import validate_protocol

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
PREPARED = ROOT / "runs/preprocessing-v1"
RAW_DEVELOPMENT = ROOT / "runs/data-v1/datasets/raw"
CHECKPOINT = ROOT / "weights/yolo26n.pt"
TRAIN_ROOT = ROOT / "runs/training-native-s10-v1"
DECISION_PATH = ROOT / "docs/issue11-v2/v3_decision.json"
RESUME = {"raw": False, "adaptive": False}
require_paths(
    protocol=PROTOCOL_PATH,
    subsets=PREPARED / "subsets.json",
    policy=PREPARED / "policy.json",
    decision=DECISION_PATH,
    training_pair=PREPARED / "datasets/S10/pair.json",
    development=RAW_DEVELOPMENT / "verified.json",
    checkpoint=CHECKPOINT,
)
protocol = validate_protocol(read_json(PROTOCOL_PATH))

<a id="pretrained-paired-inputs"></a>

## 2. Pretrained Initialization and Paired Inputs

Verify pinned pretrained semantics/architecture, verified export identities/roles, frozen recipe and equal settings. The pinned YOLO26n requires 11 backbone modules for native freeze. Pretrained inference belongs to notebook 04.

In [ ]:
experiment = prepare_pair(
    protocol,
    read_json(PREPARED / "subsets.json"),
    read_json(PREPARED / "policy.json"),
    read_json(DECISION_PATH),
    PREPARED / "datasets/S10",
    RAW_DEVELOPMENT,
    CHECKPOINT,
    TRAIN_ROOT,
)
print("Paired protocol:", experiment["protocol_id"])
print("Backbone modules frozen:", experiment["freeze"])
display(pd.Series(TRAIN_SETTINGS, name="identical branch settings"))

<a id="raw-training"></a>

## 3. Raw Training

Run or reuse the Raw branch. Existing native results are reused. To continue an interrupted run, opt in to native last.pt resume; it is not bitwise continuation.

In [ ]:
completed = {"raw": train_branch(experiment, "raw", TRAIN_ROOT, resume=RESUME["raw"])}
print("Raw completion:", completed["raw"]["budget"])

<a id="adaptive-training"></a>

## 4. Adaptive Training

Run Gamma-or-Skip independently from the same pinned initialization and full budget. Both native histories must show the same epoch/image budget; optimizer step parity is not independently audited.

In [ ]:
completed["adaptive"] = train_branch(experiment, "adaptive", TRAIN_ROOT, resume=RESUME["adaptive"])
print("Completed paired budgets:", {b: r["budget"] for b, r in completed.items()})

<a id="convergence-resources"></a>

## 5. Convergence and Resource Inspection

Inspect native training histories on the same Raw-development population. Loss decline alone is not detection improvement; distinguish best-epoch and final-epoch outcomes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for branch in ("raw", "adaptive"):
    history = pd.read_csv(TRAIN_ROOT / branch / "results.csv")
    history.plot(x="epoch", y="metrics/mAP50-95(B)", ax=axes[0], label=branch)
    history.plot(x="epoch", y="train/box_loss", ax=axes[1], label=branch)
    print(branch, "epochs:", len(history), "freeze:", experiment["freeze"])
axes[0].set_title("Shared Raw development validation")
axes[1].set_title("Native training box loss")
plt.tight_layout()
plt.show()

These plots report the current named runs only. Historical S10 results remain separately attributed, and one seed does not establish a reliable preprocessing advantage.

<a id="checkpoint-integrity"></a>

## 6. Checkpoint and Integrity Summary

Inspect native checkpoint hashes and matched epoch/image budgets. Notebook 04 loads native weights directly; it never starts a trainer.

In [ ]:
from vehicle_detection.common import file_hash

display(
    pd.DataFrame(
        [
            {
                "branch": branch,
                "best_epoch": receipt["best_epoch"],
                **receipt["budget"],
                "best_sha256": file_hash(receipt["best_checkpoint"]),
                "last_sha256": file_hash(receipt["last_checkpoint"]),
            }
            for branch, receipt in completed.items()
        ]
    )
)
print("Final test remains locked. Native epoch-boundary resume is not bitwise continuation.")